<!-- python-reference-contents -->
# Python / pandas reference

Copy snippets into your project; the small merge examples run here.

**How to read the behavior notes**

The notes describe each operation as written, using its default options unless shown otherwise.

| Label | Meaning |
|---|---|
| **Returns** | Produces a result without updating the input. Assign it to a variable if you need it later. This does not promise a deep copy of the underlying memory. |
| **Creates** | Builds an object, such as a DataFrame, model, figure, or axes. |
| **Updates** | Changes the existing named object. Some operations also return a value or the same object. |
| **Reads** | Retrieves a property, value, or selection without changing the source. |
| **Displays / checks** | Shows information or checks a condition without changing the source data. |

**Assignment matters:**

```python
sorted_df = df.sort_values('date')  # Returns a sorted table; df stays unchanged.
df = df.sort_values('date')         # Rebinds df to the returned table.
df['date'] = pd.to_datetime(df['date'])  # Updates a column in the existing df.
```

Rebinding a variable means making that name refer to the returned result; it is different from changing the original object in place. A final notebook expression can display a returned result without saving it under a name. You do not need to assign previews that you only want to inspect.

**`inplace=True` is method-specific.** Supported pandas methods such as `df.drop(...)`, `df.dropna(...)`, `df.drop_duplicates(...)`, `df.sort_values(...)`, and `df.reset_index(...)` can update the existing DataFrame with `inplace=True`; then they return `None`. Never write `df = df.dropna(inplace=True)`, because that sets `df` to `None`. `reindex()` has no `inplace` option: keep its result with assignment.

Plot methods can update an axes and return plot objects at the same time. `model.fit(...)` updates the fitted model and returns that same model. Their behavior is marked separately below.

**Contents**

- [Inspect a DataFrame](#df-inspection)
- [Select and drop columns](#select-columns)
- [Axis and negative values](#axis-negative-values)
- [Missing values](#missing-values)
- [Duplicates](#duplicates)
- [Group and summarize](#group-summary)
- [agg() versus apply(): Series, DataFrames, and groups](#agg-apply)
- [Count values and fractions: value_counts()](#value-counts)
- [Set, reset, and align indexes](#index-methods)
- [Dates and ordering](#dates-ordering)
- [Combine tables](#combining-tables)
- [Train / validation / test](#train-valid-test)
- [Preprocessing pipelines](#preprocessing-pipelines)
- [Time splits for future-return targets](#future-target-splits)
- [Training pipelines: numerical and categorical features](#mixed-feature-pipelines)
- [Compare regression model pipelines](#regression-model-pipelines)
- [Tune preprocessing and models with GridSearchCV](#pipeline-model-search)
- [Trade price and quantity outliers](#trade-outliers)
- [Histograms](#histograms)
- [Student's t-distribution: t.sf() and p-values](#t-sf)


In [1]:
# Imports pandas as pd; no DataFrame is changed.
import pandas as pd

<a id="df-inspection"></a>

## Inspect a DataFrame

| Code | Meaning | Behavior |
|---|---|---|
| `df.shape` | `(rows, columns)` | **Reads** shape tuple |
| `df.head()` | First 5 rows | **Returns** preview DataFrame |
| `df.columns.tolist()` | Column names as a list | **Returns** a list; reads column labels |
| `df.dtypes` | Type per column; `print(df.dtypes.to_string())` shows all | **Reads** dtypes Series |
| `df.info(verbose=True, show_counts=True)` | Every column's type and non-null count | **Displays** information; returns `None` |
| `df.describe()` | Numeric summary; `include="all"` includes other types | **Returns** summary DataFrame |
| `df.describe().T` | One row per summarized column; `.T` transposes | **Returns** a transposed summary |
| `df.nunique()` | Distinct non-null count per column | **Returns** counts Series |

`info()` prints and returns `None`; do not wrap it in `display()`.

**Behavior:** `.shape`, `.columns`, `.dtypes`, and `.T` are properties, not method calls. None changes `df`. `.to_string()` returns text; `print(...)`, `display(...)`, and `.info()` display information and return `None`. A preview displayed by the notebook is still a returned result, not a change to the data.


<a id="select-columns"></a>

## Select and drop columns

| Code | Meaning | Behavior |
|---|---|---|
| `df.loc[:, "TS1":"TS18"]` | Inclusive range in existing column order | **Reads** selected columns |
| `df = df.drop(columns=["col1", "col2"])` | Remove named columns | **Returns** a new table; `df =` keeps it |
| `df = df.drop(columns=df.loc[:, "TS19":"TS143"].columns)` | Remove an inclusive range | **Returns** a new table; `df =` keeps it |

**Range + extra column:**

```python
ts_columns = df.loc[:, "TS1":"TS18"].columns.tolist() + ["TS144"]
df[ts_columns]
```

`.tolist()` converts the column Index to a list; `+ ["TS144"]` returns a list with that extra name. Assign results back to save changes.

**Behavior:** `.loc[...]`, `df[ts_columns]`, and `.columns` read/select data or labels without updating `df`. `.tolist()` and list addition with `+` return lists; `+` does not append to the old list in place. `.drop()` returns a table by default. Only the assignment `df = ...` makes the name `df` refer to that result.


<a id="axis-negative-values"></a>

## Axis and negative values

For `.any()`, `axis` identifies the dimension being combined. The result is `True` if at least one checked value is `True`.

| Code | Direction checked | Result |
|---|---|---|
| `.any(axis=0)` (default) | Down the rows ↓ | One Boolean per column |
| `.any(axis=1)` | Across the columns → | One Boolean per row |

**Example: suppose `< 0` produces this table.**

| Row | bid_price | ask_price |
|---|---|---|
| 0 | False | True |
| 1 | False | False |
| 2 | True | False |

- `axis=0`: `bid_price=True`, `ask_price=True`; both columns contain a negative value.
- `axis=1`: rows `0=True`, `1=False`, `2=True`; rows 0 and 2 contain a negative value.

**Check the numeric columns in `quote`:**

```python
is_negative = quote[['bid_price', 'bid_size', 'ask_price', 'ask_size']] < 0
```

**Display complete rows containing any negative value:**

```python
negative_rows = is_negative.any(axis=1)
display(quote[negative_rows])
```

**Keep rows that do not match a condition (`~`):**

```python
zero_prices = (
    quote_clean[['ask_price', 'ask_size', 'bid_price', 'bid_size']] <= 0
).any(axis=1)

ts_quote = quote_clean.loc[~zero_prices].copy()
```

`zero_prices` marks rows containing any zero or negative price or size. `.any(axis=1)` gives one Boolean per row.

- `~` means "not": it flips `True` and `False`, so flagged rows are removed and the other rows are kept.
- `.loc[~zero_prices]` selects those rows and all columns.
- `.copy()` creates a separate DataFrame for further work.

With unique row labels, this is equivalent to the longer form:

```python
ts_quote = quote_clean.drop(index=zero_prices.index[zero_prices])
```

**Display columns containing any negative value:**

```python
negative_columns = is_negative.any(axis=0)
column_names = negative_columns.index[negative_columns]
display(quote[column_names])
```

`negative_columns.index[negative_columns]` selects the column names marked `True`. The display includes all rows of those columns.

**Common mistake:** `quote[boolean_series]` filters rows, so a Series indexed by column names cannot be used as a row filter. `quote[is_negative]` uses a Boolean DataFrame to mask individual cells, keeping the table's shape and replacing nonmatching cells with `NaN` or `NaT`.

Remember: for `.any()`, **axis 0 combines rows; axis 1 combines columns**. The other dimension remains in the result.

**Behavior:** Comparisons (`<`, `<=`), `.any()`, and `~` **return** Boolean results; they do not remove source rows. `.loc[...]`, `quote[mask]`, and `.index[mask]` **read/select** rows, cells, or labels. `.copy()` and `.drop(index=...)` **return** tables. `ts_quote = ...` stores the result under a new name; `quote_clean` stays unchanged. `display(...)` **displays** the result and returns `None`.


<a id="missing-values"></a>

## Missing values

| Code | Meaning | Behavior |
|---|---|---|
| `df.isna().sum()` | Null count per column | **Returns** counts Series |
| `df.isna().sum().sum()` | Total null cells | **Returns** a scalar count |
| `df.loc[df.isna().any(axis=1)]` | Rows with at least one null | **Reads** selected rows |
| `df.loc[:, "TS1":"TS143"].isna().all(axis=1).sum()` | Count rows with all TS1–TS143 null; excludes TS144 | **Returns** a scalar count |
| `df = df.dropna()` | Drop rows with any null | **Returns** filtered data; `df =` keeps it |
| `df = df.dropna(subset=["TS144"])` | Drop rows with a missing target | **Returns** filtered data; `df =` keeps it |
| `df = df.dropna(subset=["col1", "col2"], how="all")` | Drop rows only when both selected columns are null | **Returns** filtered data; `df =` keeps it |

`isna()` marks null cells True. Across columns (`axis=1`), `any()` means at least one True; `all()` means every selected value is True. `.sum()` counts True values.

For `dropna`, `subset` selects columns to check. `how="any"` (default) drops if any is null; `how="all"` requires all to be null. Omit `subset` to check all columns.

**Replace a missing-value code:**

```python
ts_columns = df.loc[:, "TS1":"TS144"].columns
df[ts_columns] = df[ts_columns].replace(-1, float("nan"))
```

`-1` is not null until replaced. In the news data, it means missing in TS and final popularity fields, but negative sentiment is valid. Whole-column assignment allows integer columns to become floats that hold NaN.

**Behavior:** `.isna()`, `.any()`, `.all()`, `.sum()`, and `.dropna()` **return** results without updating the source. `float('nan')` creates a scalar. `.replace(...)` also **returns** replaced data by default; in `df[ts_columns] = df[ts_columns].replace(...)`, the column assignment **updates** those columns in the existing `df`.


<a id="duplicates"></a>

## Duplicates

| Code | Meaning | Behavior |
|---|---|---|
| `df.duplicated().sum()` | Identical rows after the first; compares all columns | **Returns** a scalar count |
| `df["IDLink"].duplicated().sum()` | Repeated IDs after the first | **Returns** a scalar count |
| `df.loc[df["IDLink"].duplicated(keep=False)].sort_values("IDLink")` | Display all rows with repeated IDs | **Returns** selected, sorted rows |
| `df.loc[df.duplicated(subset=["Title", "Headline"], keep=False)].sort_values(["Title", "Headline"])` | Display repeated title/headline pairs | **Returns** selected, sorted rows |
| `df = df.drop_duplicates()` | Remove identical rows; keep the first | **Returns** deduplicated data; `df =` keeps it |
| `df = df.drop_duplicates(subset="IDLink", keep="first")` | Keep one row per ID | **Returns** deduplicated data; `df =` keeps it |

`subset` defines matching columns; all selected columns must match. Omit it to compare all columns. `duplicated()` marks rows; `drop_duplicates()` removes them.

| `keep` | `duplicated()`: marked True | `drop_duplicates()`: retained per group |
|---|---|---|
| `"first"` (default) | All except the first | First |
| `"last"` | All except the last | Last |
| `False` | All occurrences, including the first | None |

Unique rows are never marked or dropped. `.loc[...]` selects True rows; sorting puts matching values together.

**Behavior:** `.duplicated()` returns a Boolean mask, `.sum()` returns a count, and `.loc[...]` reads selected rows. `.sort_values()` and `.drop_duplicates()` return results by default. None of these calls updates the input unless a supported `inplace=True` option is explicitly used. The shown `df = ...` lines rebind `df` to the returned result.


<a id="group-summary"></a>

## Group and summarize

| Code | Meaning | Behavior |
|---|---|---|
| `df["Topic"].value_counts()` | Rows per topic, largest first | **Returns** counts Series |
| `df.groupby("Topic").size()` | Rows per topic, including missing IDLink values | **Returns** grouped counts |
| `df.groupby("Topic")["IDLink"].count()` | Non-null IDs per topic | **Returns** grouped counts |
| `df.select_dtypes(include="number").agg(["min", "max"]).T` | Minimum and maximum of each numeric column | **Returns** selected data, aggregated summary, then its transpose |

`size()` counts rows; `count()` excludes nulls in the selected column. Missing group keys are excluded by default; use `dropna=False` to include them.

**Behavior:** `.groupby()` creates a grouping object; it does not rearrange or update the source table. `.value_counts()`, `.size()`, `.count()`, `.sum()`, `.agg()`, and `.aggregate()` return summaries. `.select_dtypes()` returns selected columns; `.T` returns their transpose.

<a id="agg-apply"></a>

### `agg()` versus `apply()`: Series, DataFrames, and groups

Both methods work without `groupby()`. Use `.agg()` (also called `.aggregate()`) to calculate summaries. Use `.apply()` when you need a custom operation with a more flexible result. Both accept custom functions; `.agg()` is not limited to built-in function names.

The object before the method determines what the function receives. This table describes ordinary callable functions with the default options unless an axis is shown.

| Object and method | Function receives | Typical result |
|---|---|---|
| `series.agg(func)` | The whole Series | One summary value; multiple functions give a Series of summaries |
| `series.apply(func)` | Each individual value, by default | A Series with the original index when the function returns one scalar per value |
| `df.agg(func)` | Each column as a Series; each row with `axis=1` | Summaries per column or row |
| `df.apply(func)` | Each column as a Series; each row with `axis=1` | A Series or DataFrame, depending on what the function returns |
| `grouped.agg(func)` | Each selected column's Series within each group | One summary per group, column, and requested aggregation |
| `grouped.apply(func)` | Each whole group: a Series or DataFrame, depending on the selection | A combined Series or DataFrame; a group function can return a scalar, Series, or DataFrame |

**Small example used below:** These prices are illustrative, not results from the stock dataset.

```python
example_quotes = pd.DataFrame({
    'symbol': ['A', 'A', 'B', 'B'],
    'bid_price': [10, 11, 20, 21],
    'ask_price': [12, 15, 24, 27]
})
example_quotes['midprice'] = (
    example_quotes['bid_price'] + example_quotes['ask_price']
) / 2
```

**1. Series: one column.** Aggregation summarizes the column; ordinary `apply` calls the function on each value.

```python
midprices = example_quotes['midprice']

# Returns a Series indexed by the statistic names
price_summary = midprices.agg(['mean', 'min', 'max'])

# A custom aggregation returns one summary value
price_range = midprices.agg(lambda prices: prices.max() - prices.min())

# Returns one label for each original price
price_labels = midprices.apply(lambda price: 'high' if price > 15 else 'low')
```

Here, `price_summary` is mean 17.5, minimum 11, and maximum 24. `price_range` is 13. The labels are `low, low, high, high`; 15 is only an illustrative threshold. With modern pandas, `series.apply(func, by_row=False)` instead passes the entire Series once. The usual element-by-element behavior uses the default `by_row='compat'` for an ordinary callable.

**2. DataFrame: columns or rows.** Select the numeric price columns so the stock-symbol text is not included in arithmetic.

```python
price_columns = example_quotes[['bid_price', 'ask_price']]

# axis=0 is the default: summarize each column
column_summary = price_columns.agg(['mean', 'std'])

# axis=1: summarize each row; here the mean of bid and ask is the midpoint
row_midpoints = price_columns.agg('mean', axis=1)

# apply passes one column Series to the function at a time
column_ranges = price_columns.apply(lambda column: column.max() - column.min())

# axis=1 passes one row Series, whose index contains the column names
row_spreads = price_columns.apply(
    lambda row: row['ask_price'] - row['bid_price'], axis=1
)
```

`column_summary` has statistic names as rows and price-column names as columns. `column_ranges` has one value per column; `row_midpoints` and `row_spreads` have one value per original row. For this simple spread calculation, prefer direct column arithmetic: `price_columns['ask_price'] - price_columns['bid_price']`. DataFrame `.apply()` is not an individual-cell operation.

**3. GroupBy: operate separately within each stock.** Selecting one column creates a SeriesGroupBy; keeping several columns gives a DataFrameGroupBy.

```python
by_stock = example_quotes.groupby('symbol')

# SeriesGroupBy: summarize each stock's midpoint Series
midprice_summary = by_stock['midprice'].agg(['first', 'mean', 'std'])

# DataFrameGroupBy: name each output and choose its source column
stock_summary = by_stock.agg(
    quotes=('midprice', 'size'),
    mean_midprice=('midprice', 'mean'),
    max_ask=('ask_price', 'max')
)

# SeriesGroupBy.apply passes a whole midpoint Series for each stock
stock_ranges = by_stock['midprice'].apply(
    lambda prices: prices.max() - prices.min()
)
```

Both summary tables have one row per stock. `stock_ranges` is a Series indexed by stock. Unlike ordinary `Series.apply`, grouped `Series.apply` receives all values for one group at once. This range is also a valid `.agg()` calculation because it returns one summary value.

A DataFrameGroupBy `.apply()` can use several columns together and return complete rows. This example selects the quote with the largest spread within each stock:

```python
def widest_quote(group):
    spreads = group['ask_price'] - group['bid_price']
    return group.loc[[spreads.idxmax()]]

widest_quotes = by_stock[['bid_price', 'ask_price']].apply(widest_quote)
```

The function receives each stock's bid/ask DataFrame. The double brackets in `.loc[[...]]` retain a one-row DataFrame. The result contains stock A's row 1 and stock B's row 3, indexed by stock and original row label. This example has unique row labels; tied maximum spreads select the first matching row. Selecting columns before `.apply()` makes its inputs explicit and avoids depending on whether pandas includes the grouping column.

**For your 5-minute opening midpoint:** Sort by time, group by stock and window, then use the built-in `.first()`.

```python
q7_quote = q7_quote.sort_values('recv_time', kind='stable')
by_stock_window = q7_quote.groupby(['symbol', '5min'])
open_mid_price = by_stock_window['midprice'].first()
```

`.agg('first')` gives the same grouped summary. `.first()` skips missing values; `.apply(lambda prices: prices.iloc[0])` takes the literal first value, even if it is missing. Built-in summaries such as `.first()`, `.mean()`, and `.sum()` are usually clearer and faster than a Python function passed to `.apply()`.

**Behavior — Returns:** The shown `.agg()` and `.apply()` calls return results without updating their inputs. Assignment keeps those results. `.groupby()` creates a grouping object; it does not change the source table. Write functions that calculate and return a result without changing the row, column, or group passed into them. The example's `example_quotes['midprice'] = ...` line explicitly updates that example DataFrame.

<a id="value-counts"></a>

### Count values and fractions: `value_counts()`

Use `.value_counts()` on one column to count how often each distinct value occurs. The result is a Series: its index contains the distinct values, and its values contain their counts. Counts are sorted from largest to smallest by default.

**Example: buyer- and seller-initiated trades.** Assume `flow['direction']` contains `1` for buyers, `-1` for sellers, and missing values for unclassified trades.

```python
# Count trades in each direction
direction_counts = flow['direction'].value_counts()
print(direction_counts)

# Divide each count by the total number of classified trades
direction_fractions = direction_counts / direction_counts.sum()
print(direction_fractions)
```

For an illustrative column `[1, -1, 1, 1, -1]`, the results are:

| Direction (index) | Count | Fraction | Percentage |
|---|---:|---:|---:|
| `1` (buyer) | 3 | 0.60 | 60% |
| `-1` (seller) | 2 | 0.40 | 40% |

Each row counts once; trade quantity does not affect these fractions. The most frequent direction appears first, so the first result is not necessarily the buyer count.

**Shortcut for fractions:** `normalize=True` performs the division for you. Multiply by 100 to get percentages.

```python
direction_fractions = flow['direction'].value_counts(normalize=True)
direction_percentages = direction_fractions * 100
```

**Missing values:** By default, missing directions are excluded from both the counts and the fraction denominator. To include them as a separate category:

```python
flow['direction'].value_counts(dropna=False)
flow['direction'].value_counts(normalize=True, dropna=False)
```

With `normalize=True, dropna=False`, fractions use all rows, including unclassified trades. `.nunique()` instead returns the number of distinct nonmissing values; it does not count occurrences of each value.

**Behavior — Returns:** `.value_counts()` returns a summary without updating `flow`. Assignment keeps the returned Series. `.sum()` and division return calculated values; `print(...)` displays the result and returns `None`.

### VWAP for each stock

Assume `q2_trade` contains cleaned trades from the selected opening window, with positive prices and quantities. See [Dates and ordering](#dates-ordering) for the window filter.

For each stock, VWAP is `sum(price * quantity) / sum(quantity)`. Sum the traded value and quantity separately within each stock:

```python
q2_trade['notional'] = q2_trade['trade_price'] * q2_trade['trade_qty']  # Updates q2_trade.

by_stock = q2_trade.groupby('symbol')
VWAP = by_stock['notional'].sum() / by_stock['trade_qty'].sum()
```

- `notional`: the traded value of each row, calculated as price times quantity.
- `.groupby('symbol')`: group trades by stock.
- Each `.sum()` produces one total per stock. Division matches the totals by their stock-symbol index.
- `VWAP` is a Series: its index contains stock symbols and its values contain their VWAPs.

**Behavior:** Multiplication and division return calculated Series. Assigning `q2_trade['notional'] = ...` updates that DataFrame. The grouped sums and `VWAP = ...` create results without replacing the underlying trade rows.

**Common mistake:** `(price * quantity) / quantity.sum()` produces one contribution per trade, using all stocks' quantity in the denominator. It does not calculate each stock's VWAP. Both the numerator and denominator must be summed within the same stock.

### Find the maximum: `idxmax()` versus `max()`

```python
largest_stock = VWAP.idxmax()
largest_vwap = VWAP.loc[largest_stock]

print(largest_stock, largest_vwap)
```

| Code | Returns | Behavior |
|---|---|---|
| `VWAP.idxmax()` | Index label of the largest value: the stock symbol | **Returns** a label; source unchanged |
| `VWAP.max()` | The largest VWAP value | **Returns** a value; source unchanged |
| `VWAP.loc[largest_stock]` | The VWAP associated with that stock label | **Reads** the value at that label |
| `VWAP.nlargest(1)` | A one-row Series showing the stock label and largest value | **Returns** a smaller Series; source unchanged |

`idxmax()` returns the first matching label if several stocks share the maximum. Finding the maximum does not require sorting.

### Index labels versus positions: `.iloc[-1]` versus `[-1]`

Use `.loc[label]` to select by index label and `.iloc[position]` to select by row position. `.iloc[0]` selects the first row; `.iloc[-1]` selects the last.

```python
sorted_vwap = VWAP.sort_values()  # Returns a sorted Series; VWAP stays unchanged.
largest_vwap = sorted_vwap.iloc[-1]  # Reads the final value.
```

`sort_values()` sorts values in ascending order by default, so the last row contains the largest value when there are no missing values. It retains the original stock-symbol labels. `.iloc[-1]` returns the value at the last position regardless of its label. Missing values sort last by default; `.max()` skips them instead.

With the stock-symbol index in your current pandas environment, `VWAP.sort_values()[-1]` raises `KeyError`: `[-1]` searches for the index label `-1`, not the last position. Use `.iloc[-1]` to make positional selection explicit.

`.iloc[-1]` alone means the last value in the current order, not necessarily the maximum. Prefer `.max()` for the maximum value and `.idxmax()` for its stock symbol.

### Trade volume in 30-minute windows

Give each trade a window label, then sum quantities for each window-stock pair. This example assumes a nonempty, one-day `trade_clean` with `recv_time` already converted to Shanghai datetimes.

**1. Filter the time range.** Each window includes its start and excludes its end: 09:30 <= time < 10:00, 10:00 <= time < 10:30, and so on. With this convention, exactly 15:00 is excluded.

```python
day = trade_clean['recv_time'].iloc[0].normalize()
q3_start = day + pd.Timedelta('9h30min')
q3_end = day + pd.Timedelta('15h')

q3_trade = trade_clean.loc[
    trade_clean['recv_time'].between(
        q3_start, q3_end, inclusive='left'
    )
].copy()
```

**2. Label each trade with its window's start.**

```python
q3_trade['window'] = q3_trade['recv_time'].dt.floor('30min')  # Updates q3_trade.
q3_trade[['recv_time', 'window', 'symbol', 'trade_qty']].head()
```

`.dt` provides datetime operations; it does not convert or modify the column itself. `.floor('30min')` **returns** rounded timestamps without changing `recv_time`; assigning the result to `q3_trade['window']` **updates** that DataFrame. It rounds down to the nearest :00 or :30 while preserving the date and timezone. For example, 09:47 becomes 09:30, and exactly 10:00 belongs to the next window, labelled 10:00.

Illustrative trades, with dates and timezones omitted from the table for readability:

| Trade time | Symbol | Quantity | Window label |
|---|---|---:|---|
| 09:32 | A | 100 | 09:30 |
| 09:47 | A | 200 | 09:30 |
| 09:55 | B | 50 | 09:30 |
| 10:00 | A | 80 | 10:00 |

**3. Group by both window and stock, then sum.**

```python
by_window_stock = q3_trade.groupby(['window', 'symbol'])

volume = by_window_stock['trade_qty'].sum()  # Returns a grouped Series.
volume = volume.reset_index(name='volume')  # Returns a DataFrame; assignment keeps it.
```

- `groupby(['window', 'symbol'])`: rows belong to the same group only when both labels match.
- `['trade_qty'].sum()`: sum the traded quantities in each group. This measures volume, not the number of trades.
- Before `reset_index`, the result is a Series with a two-level index: `window` and `symbol`.
- `reset_index(name='volume')` moves both index levels into ordinary columns, names the summed values `volume`, and gives the DataFrame a new 0, 1, 2, ... index. Here, `name` names the Series values; it does not rename the grouping keys.
- Do not use `drop=True` here: you want to retain the window and stock labels as columns.

The illustrative trades become:

| window | symbol | volume |
|---|---|---:|
| 09:30 | A | 300 |
| 09:30 | B | 50 |
| 10:00 | A | 80 |

Grouping only by `symbol` combines all windows. Grouping only by `window` combines all stocks. Use both keys for a separate total for each stock within each window.

**4. Find the stock with the largest volume in each window.**

Start with `volume`, the table containing one row per window-stock pair:

```python
largest_rows = volume.groupby('window')['volume'].idxmax()
leaders = volume.loc[largest_rows]  # Reads selected rows; volume stays unchanged.
leaders  # The notebook displays this expression; it does not change the table.
```

- `.groupby('window')`: group the stock totals within each window. The previous step summed individual trades by both window and stock; this step compares those stock totals within each window.
- `['volume']`: choose the numeric column used to select the winner.
- `.idxmax()`: return the original row label with the largest volume in each group, not the volume itself.
- `volume.loc[largest_rows]`: retrieve those complete rows, including the stock symbol, window, and volume.

**Why the extra `.loc` step?** `VWAP.idxmax()` returned a stock symbol because the VWAP Series was indexed by stock. After `reset_index(name='volume')`, the `volume` DataFrame has integer row labels, so `idxmax()` returns those row labels. `.loc` uses them to recover the winning stocks.

Labels such as `312` and `313` in the displayed result are retained from `volume`; they are not ranks or window numbers. Optionally renumber the result for display:

```python
leaders = leaders.reset_index(drop=True)  # Returns a result; assign it to keep the new index.
```

`volume.groupby('window')['volume'].max()` returns only the largest quantity in each window, without identifying the stock. Calling `.idxmax()` on the whole grouped DataFrame would find a maximum separately for every remaining column; select `['volume']` first.

If stocks tie for the highest volume, `idxmax()` selects the first matching row in the current order.

Windows with no trades do not appear in this result. Add the full set of windows explicitly when reporting every half-hour, including empty lunch periods.

<a id="index-methods"></a>

### `set_index()`, `reset_index()`, and `reindex()`

| Method | What it does | Missing rows | Behavior by default |
|---|---|---|---|
| `df.set_index('minutes')` | Uses an existing column as the row index; removes that column from the data columns by default | Does not add rows | **Returns** a DataFrame; original unchanged |
| `vol.reset_index(name='volume')` | Moves the Series index into ordinary columns, names its values `volume`, and creates a numbered index | Does not add rows | **Returns** a DataFrame; original Series unchanged |
| `vol.reindex(all_minutes, fill_value=0)` | Matches existing values to the requested index labels and their order | Adds absent labels with zero here; omits labels outside the requested list | **Returns** an aligned Series; original unchanged |

`set_index()` changes what identifies each row. `reset_index()` puts those identifiers back into columns. `reindex()` chooses which identifiers appear and their order. Setting an index does not sort or aggregate the data.

Keep the return value with assignment. DataFrame `set_index()` and `reset_index()` support `inplace=True` and then return `None`. A Series cannot be converted into a DataFrame in place: `vol.reset_index(name='volume')` needs assignment; Series `reset_index(inplace=True)` is only supported with `drop=True`. `reindex()` has no `inplace` option.


`reindex()` **returns** a Series or DataFrame with the requested labels and order. `reset_index()` **returns** a result with the old index moved into columns and a new numbered index. Neither updates the original by default.

Start with a Series whose index contains the window timestamps:

```python
volume_by_window = volume.groupby('window')['volume'].sum()
```

**`reindex()`: align the data to the labels you provide.**

```python
all_windows = pd.date_range(
    q3_start, q3_end, freq='30min', inclusive='left', name='window'
)
complete_volume = volume_by_window.reindex(all_windows, fill_value=0)
```

Existing values are matched by timestamp label. Requested labels absent from the original Series get a new row; here, `fill_value=0` gives them zero recorded volume. Without it, new values are `NaN`. Labels outside the requested list are omitted, and the result follows the list's order. `reindex()` does not change the original Series; assigning its return value to `complete_volume` keeps the result under a separate name.

`name='window'` names the timestamp index, so resetting it later produces a column called `window`. `fill_value` fills newly introduced labels; it does not replace missing values already present in the original data.

**`reset_index()`: change how the existing labels are stored.**

```python
volume_table = volume_by_window.reset_index(name='volume')
```

This converts the original Series to a DataFrame with `window` and `volume` columns and a new 0, 1, 2, ... index. It does not add missing windows. Here, `name='volume'` names the Series values, not the index column.

For illustration, suppose the original Series has only these rows (dates and timezones omitted):

| window (index) | volume |
|---|---:|
| 09:30 | 100 |
| 10:30 | 200 |

Reindexing to 09:30, 10:00, and 10:30 with `fill_value=0` gives:

| window (index) | volume |
|---|---:|
| 09:30 | 100 |
| 10:00 | 0 |
| 10:30 | 200 |

Resetting the original Series instead gives:

| new index | window | volume |
|---|---|---:|
| 0 | 09:30 | 100 |
| 1 | 10:30 | 200 |

The missing 10:00 window is still absent after resetting.

| Code | Purpose | Behavior |
|---|---|---|
| `.reindex(all_windows, fill_value=0)` | Include exactly the requested windows in their given order | **Returns** aligned Series/DataFrame; assign to keep |
| `.reset_index(name='volume')` | Convert the grouped Series to a table; name its values `volume` | **Returns** DataFrame from Series; assign to keep |
| `.reset_index(drop=True)` | Discard the old index labels and renumber the rows | **Returns** Series/DataFrame with new index by default |

For the earlier Series grouped by both window and stock, `reset_index(name='volume')` moves both index levels into columns, giving `window`, `symbol`, and `volume`.

You can use both operations in sequence: `complete_table = complete_volume.reset_index(name='volume')` keeps a table that includes the previously missing windows. `complete_volume` itself remains a Series.

**Keep the returned result.** This call alone leaves the original Series unchanged:

```python
volume_by_window.reindex(all_windows, fill_value=0)  # Returns a result, but does not save it.
```

To use the added windows in later code, assign the result back:

```python
volume_by_window = volume_by_window.reindex(all_windows, fill_value=0)
```

Without the assignment, later labels and plots still use the old index, so missing lunch windows remain absent. Alternatively, use a new variable such as `complete_volume` and use that variable in later calculations. `reindex()` has no `inplace=True` option.

### Apply the three methods to per-minute volume

After grouping by `symbol` and `minutes` and calling `reset_index(name='volume')`, `by_minute_vol` has three columns: `symbol`, `minutes`, and `volume`. Its row index is numbered. The raw `recv_time` column is no longer present.

Assume `stock` is one symbol from your top-five list, and `minutes` contains Shanghai datetime values. Select that stock before setting the time index, so there is one row per minute.

```python
all_minutes = pd.date_range(
    q3_start, q3_end, freq='min', inclusive='left', name='minutes'
)

stock_rows = by_minute_vol.loc[
    by_minute_vol['symbol'] == stock, ['minutes', 'volume']
]

# Returns a table whose row labels are timestamps.
indexed_rows = stock_rows.set_index('minutes')
vol = indexed_rows['volume']  # Reads the volume column as a Series.

# Returns a result aligned to the complete minute grid.
complete_vol = vol.reindex(all_minutes, fill_value=0)

# Returns a table with minutes and volume as ordinary columns.
volume_table = complete_vol.reset_index(name='volume')
```

For illustration, if the original stock has volume 100 at 09:30 and 300 at 09:32, reindexing adds 09:31 with zero volume. `reset_index()` then moves those three timestamps into the `minutes` column. All original inputs remain unchanged in this example because the results use separate variable names.

**Common plotting mistakes:**

- Use `freq='min'` or `'1min'`, not `'minute'`. `inclusive='left'` includes 09:30 and excludes exactly 15:00; use the same boundary convention when filtering trades.
- Select `minutes`, not `recv_time`, from the aggregated table.
- Set the timestamp index before reindexing. `reindex()` matches the existing row labels, not a time column it finds inside the table. Integer row labels do not match timestamps. Reindexing a numeric Series with the wrong index can produce misleading zeros; applying `fill_value=0` to a table that still has a datetime column can instead raise a type error.
- `labels = complete_vol.index.strftime('%H:%M')` **returns** strings without modifying the index. That variable has no effect unless the plotting code uses it. For a datetime x-axis, format the ticks with `matplotlib.dates.DateFormatter('%H:%M', tz=all_minutes.tz)`.

For the plot, `axes[i].plot(complete_vol.index, complete_vol / 1_000_000)` uses timestamps on the x-axis and volume in millions on the y-axis. It **updates** the axes and **returns** line objects. Filling absent trade-minute groups with zero represents no recorded volume; it does not establish that the underlying feed was complete.


<a id="dates-ordering"></a>

## Dates and ordering

| Code | Meaning | Behavior |
|---|---|---|
| `df["date"] = pd.to_datetime(df["date"])` | Convert date strings to datetime | **Returns** converted values; column assignment **updates** df |
| `df = df.sort_values("date")` | Oldest first; `ascending=False` gives newest first | **Returns** sorted data; assignment rebinds df |
| `df = df.sort_values("date", kind="stable")` | Oldest first; equal dates keep their original relative order | **Returns** sorted data; assignment rebinds df |
| `df = df.reset_index(drop=True)` | Renumber the index from 0; discard old index labels | **Returns** a result with new index; assignment rebinds df |

Sorting moves whole rows. `reset_index(drop=True)` does not delete data rows or change ID columns.

**Example: convert publish dates and sort articles.**

```python
df["PublishDate"] = pd.to_datetime(df["PublishDate"])
df = df.sort_values(["PublishDate", "Title"]).reset_index(drop=True)
df.head()
```

`PublishDate` is the primary sort key; `Title` breaks ties when timestamps match. Both sort in ascending order by default. `head()` previews the first five rows.

**Example: convert microsecond timestamps to Shanghai time.**

```python
result['time'] = pd.to_datetime(result['recv_time'], unit='us', utc=True).dt.tz_convert('Asia/Shanghai')
result = result.sort_values('time', kind='stable').reset_index(drop=True)
```

- `unit='us'`: interpret `recv_time` as microseconds since `1970-01-01 00:00:00 UTC` (the Unix epoch).
- `utc=True`: create datetimes with the UTC time zone. UTC means Coordinated Universal Time, the global reference for time zones.
- `.dt.tz_convert('Asia/Shanghai')`: express the same moments in Shanghai time (UTC+8). `.dt` provides datetime operations for a pandas column.
- `result['time'] = ...`: create or replace `time`; keep the original `recv_time` column.

For example, `2026-10-28 08:00:00+00:00` in UTC is `2026-10-28 16:00:00+08:00` in Shanghai. The clock time changes, but the moment stays the same.

**Stable timestamp sorting:** When sorting one timestamp column, `kind='stable'` keeps rows with equal timestamps in their existing relative order. The default sort does not guarantee this; stable sorting does not remove duplicates.

Convert timestamps first, then sort before calculations that depend on event order, such as `diff()`, `shift()`, or quote matching. Filtering preserves row order, so filtering an already sorted table does not require another sort.

**Example: filter a window within one trading day.**

`recv_time` contains full timestamps, such as `2020-10-09 09:35:00+08:00`. `pd.Timedelta('9h30min')` is a duration of 9 hours 30 minutes, not a timestamp. Comparing these directly raises an error; add the duration to the trading date first.

For a nonempty, one-day dataset with `recv_time` already converted to Shanghai datetimes:

```python
day = trade_clean['recv_time'].iloc[0].normalize()

q2_start = day + pd.Timedelta('9h30min')
q2_end = day + pd.Timedelta('10h')

q2_trade = trade_clean.loc[
    trade_clean['recv_time'].between(q2_start, q2_end)
].copy()
```

- `.iloc[0]`: select the first timestamp.
- `.normalize()`: set its time to midnight while preserving its date and timezone.
- Adding each duration gives a full timestamp that can be compared with `recv_time`.
- `.between(q2_start, q2_end)`: select timestamps within the window; both endpoints are included by default.
- `.copy()`: make a separate table for the selected trades.

```text
day       -> 2020-10-09 00:00:00+08:00
q2_start  -> 2020-10-09 09:30:00+08:00
q2_end    -> 2020-10-09 10:00:00+08:00
```

**Behavior:** `pd.to_datetime()`, `.dt.tz_convert()`, `.normalize()`, and `.dt.floor()` **return** converted values; the original timestamp or column stays unchanged. `pd.Timedelta(...)` and `pd.date_range(...)` **create** duration or datetime-index objects. Datetime addition returns new timestamps; `.between()` returns a Boolean mask.

`.head()`, `.loc[...]`, and `.iloc[...]` **read/return** selected data; `.copy()` creates a separate object. `.sort_values()`, `.reset_index()`, `.diff()`, and `.shift()` **return** results by default. Column assignments such as `df['PublishDate'] = ...` and `result['time'] = ...` **update** those DataFrames; `df = ...`, `day = ...`, and `q2_trade = ...` retain returned results under variable names.

`.index` reads the index. `.index.strftime('%H:%M')` **returns** formatted string labels; it does not change the datetime index. Store those labels with `labels = ...` if you will use them in a chart.


<a id="combining-tables"></a>

## Combine tables

**`merge`: match keys.** `on` names the key; `how` selects the keys to retain. Unmatched values become null.

| `how` | Retains |
|---|---|
| `"inner"` (default) | Intersection of keys |
| `"left"` / `"right"` | Left / right keys |
| `"outer"` | Union of keys |
| `"cross"` | Every pair of rows; omit `on` |

**Behavior:** `pd.DataFrame(...)` **creates** a table. `.merge(...)` **returns** a combined DataFrame without changing either input. The `how` option affects that returned table only.


In [2]:
# Constructors create two DataFrames.
left = pd.DataFrame({"id": [1, 2], "name": ["Amy", "Ben"]})
right = pd.DataFrame({"id": [2, 3], "score": [80, 90]})

# Returns a new table; the final expression is displayed, not assigned.
left.merge(right, on="id", how="outer", indicator=True, validate="one_to_one")

,id,name,score,_merge
0,1,Amy,NaN,left_only
1,2,Ben,80.0,both
2,3,NaN,90.0,right_only


| Merge option | Purpose |
|---|---|
| `on=["id", "date"]` | Match multiple columns |
| `left_on="id", right_on="user_id"` | Different key names |
| `suffixes=("_old", "_new")` | Label overlapping non-key columns |
| `indicator=True` | Add `_merge`: `left_only`, `right_only`, `both` |
| `validate="one_to_one"` | Both keys unique; `many_to_one` requires only right keys unique; `one_to_many` only left |

Duplicate keys can multiply rows (2 × 3 matches = 6 rows). Check with `left["id"].duplicated().sum()` and `left["id"].isna().sum()`; repeat for `right`. Pandas matches null keys to each other.

**Stack or align tables:**

| Code | Meaning | Behavior |
|---|---|---|
| `pd.concat([left, right], ignore_index=True)` | Stack rows; align column names; reset index | **Returns** a combined DataFrame; inputs unchanged |
| `pd.concat([left, right], axis=1)` | Combine columns by existing row index | **Returns** an aligned DataFrame; inputs unchanged |
| `left.set_index("id").join(right.set_index("id"))` | Match indexes; left join by default | **Returns** indexed tables, then a joined table |

`concat` does not remove duplicates. For `axis=1`, set meaningful indexes first if row positions do not identify the same records.

**Behavior:** `.set_index()` and `.join()` return results by default; the original tables stay unchanged. `ignore_index`, `suffixes`, and `indicator` affect only the output. `validate` checks the relationship and can raise an error. `.duplicated()`, `.isna()`, and `.sum()` return validation results, not data changes.

**`merge_asof`: match nearby times or numbers.** Both tables must be sorted ascending by the matching key; at most one right match per left row. It **returns** a matched DataFrame and leaves both inputs unchanged.


In [3]:
# Constructors create DataFrames; sorting returns results saved by assignment.
events = pd.DataFrame({"time": [3]})
prices = pd.DataFrame({"time": [0, 5], "price": [100, 102]})
events = events.sort_values("time")
prices = prices.sort_values("time")

# Returns a matched table; assign it to a name if you need it later.
pd.merge_asof(events, prices, on="time", direction="backward")  # price 100

,time,price
0,3,100


`direction`: `"backward"` (default) → latest ≤ key; `"forward"` → earliest ≥ key; `"nearest"` → closest.

`tolerance=2` limits the gap to 2 units (use `pd.Timedelta("2min")` for datetime keys); `by="symbol"` matches within groups; `allow_exact_matches=False` excludes equal keys. For forecasting, match backward using the time information became available.

Docs: [merge](https://pandas.pydata.org/docs/reference/api/pandas.merge.html) · [concat](https://pandas.pydata.org/docs/reference/api/pandas.concat.html) · [join](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html) · [merge_asof](https://pandas.pydata.org/docs/reference/api/pandas.merge_asof.html)

**Behavior:** `pd.Timedelta(...)` creates a duration. The matching options control the returned as-of table; they do not update either input. In the preceding code cell, the notebook displays the unassigned final result. Use `matched = pd.merge_asof(...)` if later cells need that table.


<a id="train-valid-test"></a>

## Train / validation / test

Assumes cleaned `df` with non-null `PublishDate`, `Title`, and `TS144`. Keep time order and prevent exact-title overlap.

```python
df["PublishDate"] = pd.to_datetime(df["PublishDate"])
df = df.sort_values("PublishDate").reset_index(drop=True)

# Approximate 70% / 15% / 15% before exclusions
valid_start = df["PublishDate"].iloc[int(len(df) * 0.70)]
test_start = df["PublishDate"].iloc[int(len(df) * 0.85)]

# Conservative assumption: 48-hour target + 20-minute collection buffer
label_ready = df["PublishDate"] + pd.Timedelta(hours=48, minutes=20)

train = df.loc[label_ready < valid_start].copy()
valid = df.loc[(df["PublishDate"] >= valid_start) & (label_ready < test_start)].copy()
test = df.loc[df["PublishDate"] >= test_start].copy()

# Exclude titles already used in an earlier split
valid = valid.loc[~valid["Title"].isin(train["Title"])].copy()
test = test.loc[~test["Title"].isin(train["Title"])].copy()
test = test.loc[~test["Title"].isin(valid["Title"])].copy()
```

`.isin()` finds matching titles; `~` keeps non-matches. Matching is exact. Excluded rows remain in `df`; split sizes shrink. Adjust the label-availability assumption for other targets.

**Checks:** `assert` raises an error if a condition is false. Keep split indexes unchanged so `label_ready.loc[...]` matches the original rows.

```python
assert len(train) > 0 and len(valid) > 0 and len(test) > 0

assert label_ready.loc[train.index].max() < valid["PublishDate"].min()
assert label_ready.loc[valid.index].max() < test["PublishDate"].min()

assert set(train["Title"]).isdisjoint(valid["Title"])
assert set(train["Title"]).isdisjoint(test["Title"])
assert set(valid["Title"]).isdisjoint(test["Title"])
```

`isdisjoint()` means no shared values. Fit preprocessing/model on train; tune on validation; evaluate the final choice on test. For this task, use metadata + TS1–TS18 as inputs; exclude TS144 (the target), TS19–TS143, and final platform counts.

**Behavior:**

- **Returns:** `pd.to_datetime()`, `.sort_values()`, `.reset_index()`, datetime arithmetic, `.isin()`, comparisons, and Boolean operators. `.copy()` creates each split table; the filtering expressions do not remove rows from `df`.
- **Updates:** `df['PublishDate'] = ...` changes a column. In contrast, `df = ...`, `valid = ...`, and `test = ...` rebind names to returned tables.
- **Reads / creates:** `.loc`, `.iloc`, and `.index` retrieve selections or labels; `pd.Timedelta(...)` creates a duration. `len()`, `int()`, `.min()`, and `.max()` return values; `set(...)` creates a set and `.isdisjoint()` returns a Boolean.
- **Checks:** `assert` raises an error when a condition fails; it does not clean or update the data.


<a id="preprocessing-pipelines"></a>

## Preprocessing pipelines

`ColumnTransformer` applies different preprocessing to different columns. `Pipeline` runs steps in order and keeps preprocessing attached to the model.

**Example: predict 48-hour shares using information available at six hours.** Start with the cleaned `train`, `valid`, and `test` above. Replace the TS missing-value code `-1` with NaN first; observed zero shares are valid. Targets must be observed.

**1. Construct features and targets.** Use all 18 early counts, two sentiment scores, topic, and source. Earlier counts describe the trajectory; `TS18` describes popularity at six hours.

```python
import numpy as np

early_columns = [f"TS{i}" for i in range(1, 19)]
numeric_columns = early_columns + ["SentimentTitle", "SentimentHeadline"]
categorical_columns = ["Topic", "Source"]

def make_features(data):
    features = data[numeric_columns + categorical_columns].copy()
    features[early_columns] = np.log1p(features[early_columns])
    return features

X_train = make_features(train)
X_valid = make_features(valid)
X_test = make_features(test)

y_train = train["TS144"].copy()
y_valid = valid["TS144"].copy()
y_test = test["TS144"].copy()
```

`log1p` compresses large counts, preserves zeros, and leaves NaNs missing. Sentiment scores can be negative, so this count transformation is not applied to them. Raw `Title`, `Headline`, and `IDLink` are excluded from these inputs. Keep `TS144`, later counts, and anything calculated from the target out of `X`.

**Behavior:** `make_features()` **returns** a copied feature table. Its `features[early_columns] = ...` assignment **updates only that copy**. `.copy()` creates the target Series. `np.log1p()` returns transformed values as used here; it does not change the original inputs. List comprehensions and list addition create lists; `range(...)` supplies the values used to construct the column names.

**2. Define preprocessing and the model.** Nothing is fitted when these objects are created.

```python
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge

numeric_steps = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler())
])

categorical_steps = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encode", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_steps, numeric_columns),
    ("categorical", categorical_steps, categorical_columns)
])

model = Pipeline([
    ("preprocess", preprocessor),
    ("regression", Ridge(alpha=1.0, solver="lsqr"))
])
```

| Step / option | Meaning |
|---|---|
| `SimpleImputer(strategy="median")` | Fill each numeric column using its training median, after the count log transformation |
| `add_indicator=True` | Add a missingness flag for each column that contained missing values during fitting |
| `StandardScaler()` | Center and scale numeric inputs using training statistics |
| `fill_value="Unknown"` | Replace missing categorical values with a named category |
| `OneHotEncoder()` | Create an indicator column for each learned category |
| `handle_unknown="ignore"` | An unseen category gets zeros across that feature's one-hot columns |
| `Ridge(alpha=1.0, solver="lsqr")` | Regularized linear regression; this solver supports the sparse encoded inputs |

**Behavior — Creates:** `SimpleImputer(...)`, `StandardScaler(...)`, `OneHotEncoder(...)`, `Ridge(...)`, `ColumnTransformer(...)`, and `Pipeline(...)` construct unfitted objects. Their parameters configure what a later fit/transform/predict will do; construction alone does not change the input data.

Each `ColumnTransformer` tuple is `(name, transformer, columns)`; each `Pipeline` tuple is `(name, step)`. Names label steps and can be chosen freely. Unlisted columns are dropped by `ColumnTransformer` by default.

Before scaling, a missingness flag is 1 for a missing value and 0 for an observed value. The median is a computational placeholder, not a reconstruction of the true count. Do not replace unobserved counts with zero. Scaling helps Ridge penalize features on comparable scales; Ridge is useful when the cumulative TS features are strongly related.

**3. Fit on training data; evaluate on validation data.** Keep `y` in share units, but fit the model to `log1p(y_train)`. `expm1` converts predictions back to shares.

```python
from sklearn.metrics import root_mean_squared_log_error

model.fit(X_train, np.log1p(y_train))

valid_predictions = np.expm1(model.predict(X_valid))
valid_predictions = np.maximum(valid_predictions, 0)

# Baseline: no additional shares after six hours
baseline_predictions = valid["TS18"]

baseline_rmsle = root_mean_squared_log_error(y_valid, baseline_predictions)
model_rmsle = root_mean_squared_log_error(y_valid, valid_predictions)

print(f"Baseline validation RMSLE: {baseline_rmsle:.4f}")
print(f"Ridge validation RMSLE:    {model_rmsle:.4f}")
```

`model.fit(...)` learns medians, scaling statistics, categories, and regression coefficients from training data. `model.predict(...)` reuses those fitted steps; it does not refit them on validation data. RMSLE measures errors on the `log(1 + shares)` scale; lower is better. This baseline requires observed `TS18` values. Use validation to choose features and settings; evaluate the final choice on test once.

**Behavior:**

| Operation | Existing object / returned result |
|---|---|
| `model.fit(...)` | **Updates** the fitted model and its fitted steps; **returns the same model**. No reassignment is required. With the defaults shown, the input tables stay unchanged. |
| `model.predict(...)` | **Returns** predictions using the fitted model; does not refit it. Save predictions with assignment. |
| `np.log1p(...)`, `np.expm1(...)`, `np.maximum(...)` | **Return** calculated outputs as written, with no `out=` argument; inputs remain unchanged. |
| `valid['TS18']` | **Reads** a Series; it does not update `valid`. |
| `root_mean_squared_log_error(...)` | **Returns** a numeric score; data and model stay unchanged. |
| `print(...)` | **Displays** text; returns `None`. |

`valid_predictions = np.maximum(valid_predictions, 0)` keeps the returned clipped values by rebinding the variable. Calling `np.maximum(valid_predictions, 0)` without assignment would leave the variable's existing values unchanged.


<a id="future-target-splits"></a>

## Time splits for future-return targets

For Q8, start with `model_data` containing features, `recv_time`, `target_time = recv_time + 10 seconds`, and the forward-return `target`. Build the target separately from the training pipeline; it must never become an input feature.

Reserve the latest approximately 40% of rows for test. Use the last 20% of the earlier development period for validation. Keep equal timestamps together and exclude labels that reach a later split.

```python
ordered = model_data.sort_values("recv_time").reset_index(drop=True)
test_start = ordered.iloc[int(0.6 * len(ordered))]["recv_time"]

development = ordered.loc[ordered["target_time"] < test_start].copy()
test = ordered.loc[ordered["recv_time"] >= test_start].copy()

valid_start = development.iloc[int(0.8 * len(development))]["recv_time"]
train = development.loc[development["target_time"] < valid_start].copy()
valid = development.loc[development["recv_time"] >= valid_start].copy()

assert len(train) > 0 and len(valid) > 0 and len(test) > 0
assert train["target_time"].max() < valid["recv_time"].min()
assert valid["target_time"].max() < test["recv_time"].min()

print("Split sizes:", len(train), len(valid), len(test))
```

The final pipeline can be refitted on `train` + `valid` after model selection. Boundary exclusions make the proportions approximate. For other tasks, split using the time the outcome becomes available, and apply any required entity/group exclusions first. `TimeSeriesSplit(gap=10)` excludes ten **rows**, not ten seconds; irregular timestamps need time-based exclusions like these.

**Behavior:** sorting and filtering return tables; `.copy()` creates independent split tables. These assignments do not remove rows from `model_data`. `assert` checks the boundaries without changing data.

Docs: [TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html).


<a id="mixed-feature-pipelines"></a>

## Training pipelines: numerical and categorical features

This is a separate, reusable regression template. Start with already separated `train`, `valid`, and `test` tables. The column names below match Q8; use the [future-target split](#future-target-splits) above for that task. For another dataset, change the feature lists and target name.

### Select the inputs

```python
import numpy as np
import pandas as pd

numeric_columns = [
    "relative_spread", "imbalance", "log_depth",
    "trade_gap", "log_trade_qty", "has_recent_trade",
]
categorical_columns = ["symbol"]
target_column = "target"
feature_columns = numeric_columns + categorical_columns

def make_model_features(data):
    features = data[feature_columns].copy()
    categories = features[categorical_columns].astype("string").astype(object)
    features[categorical_columns] = categories.where(categories.notna(), np.nan)
    return features

X_train = make_model_features(train)
X_valid = make_model_features(valid)
X_test = make_model_features(test)

y_train = train[target_column].copy()
y_valid = valid[target_column].copy()
y_test = test[target_column].copy()

assert target_column not in feature_columns
assert not set(numeric_columns).intersection(categorical_columns)
assert np.isfinite(y_train).all()
assert np.isfinite(y_valid).all()
assert np.isfinite(y_test).all()

print("Input shapes:", X_train.shape, X_valid.shape, X_test.shape)
display(X_train.head(3))
```

Numerical columns must be numeric and contain finite values or `np.nan`. `make_model_features()` copies each dataset, converts categories (including numeric category codes) to strings, and normalizes missing markers such as `None` or `pd.NA` to `np.nan`. It leaves the original table unchanged. Use consistent category definitions across splits. Omit raw timestamps, identifiers without a useful modeling meaning, and future/target-derived columns. An empty numerical or categorical list is allowed; at least one list must contain features.

### Define preprocessing once

```python
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge

numeric_steps = Pipeline([
    ("impute", SimpleImputer(
        strategy="median", add_indicator=True, keep_empty_features=True
    )),
    ("scale", StandardScaler()),
])

categorical_steps = Pipeline([
    ("impute", SimpleImputer(
        strategy="constant", fill_value="Missing", keep_empty_features=True
    )),
    ("encode", OneHotEncoder(
        handle_unknown="ignore", sparse_output=False
    )),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_steps, numeric_columns),
        ("categorical", categorical_steps, categorical_columns),
    ],
    remainder="drop",
    sparse_threshold=0,
)

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", Ridge(alpha=1.0)),
])
```

| Component | What it learns or does |
|---|---|
| Numerical imputer | Learns fitting-set medians; adds flags for columns missing during fitting |
| `keep_empty_features=True` | Retains a column even if every fitting value is missing; an all-missing numeric column is filled with zero |
| Numerical scaler | Learns fitting-set means and scales; useful for regularized linear models |
| Categorical imputer | Uses the named `Missing` category for missing values |
| One-hot encoder | Learns categories; an unseen category produces all zeros for that feature's encoded columns |
| `ColumnTransformer` | Combines the two branches and drops unlisted columns |
| `Pipeline` | Fits preprocessing and the model together, then reuses them during prediction |

The dense output is suitable for a modest number of categories and preserves observed zeros for XGBoost. Many distinct categories can make it large. XGBoost's tree booster treats absent sparse entries as missing, so changing to sparse output can change the meaning of numeric zeros. See the [XGBoost FAQ](https://xgboost.readthedocs.io/en/stable/faq.html#why-do-i-see-different-results-with-sparse-and-dense-data).

### Fit a single model

```python
pipeline.fit(X_train, y_train)
valid_predictions = pipeline.predict(X_valid)

feature_names = pipeline.named_steps["preprocess"].get_feature_names_out()
print("Encoded feature count:", len(feature_names))
print("First feature names:", feature_names[:8])
```

**Behavior:** constructors create unfitted objects. `pipeline.fit(...)` updates the pipeline and its steps, returning that same pipeline; the input tables remain unchanged with these defaults. `predict(...)` returns predictions without refitting. Never fit the preprocessor on all data before validation. Keep the target on its original scale here: signed returns do not use the share-count example's `log1p`/`expm1` treatment.

Docs: [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) · [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) · [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).


<a id="regression-model-pipelines"></a>

## Compare regression model pipelines

Reuse `pipeline`, `X_train`, `X_valid`, and the targets from the mixed-feature setup. These are starting settings, not claims about the best model for a dataset.

### Choose candidate models

```python
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

model_choices = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Linear regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Elastic net": ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000),
    "Random forest": RandomForestRegressor(
        n_estimators=100, max_depth=8, min_samples_leaf=10,
        random_state=42, n_jobs=1,
    ),
}
```

**Optional XGBoost:** run this block before the comparison if `xgboost` is installed in the notebook's Python environment. The other models need only scikit-learn.

```python
from xgboost import XGBRegressor

model_choices["XGBoost"] = XGBRegressor(
    objective="reg:squarederror", tree_method="hist",
    n_estimators=150, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=42, n_jobs=1,
)
```

| Model | Useful starting role | Main settings |
|---|---|---|
| Mean baseline | Check whether features improve on a constant prediction | None |
| Linear regression | Unpenalized linear benchmark | No regularization parameter |
| Ridge | Shrink coefficients, especially with correlated features | `alpha` |
| Elastic net | Combine L1 and L2 penalties; can set coefficients to zero | `alpha`, `l1_ratio` |
| Random forest | Average trees to model nonlinear effects and interactions | `n_estimators`, `max_depth`, `min_samples_leaf`, `max_features` |
| XGBoost | Add trees sequentially to reduce prediction errors | `n_estimators`, `learning_rate`, `max_depth`, `min_child_weight`, `reg_lambda` |

Full one-hot columns plus an intercept can make ordinary linear regression coefficients non-unique. For interpretable OLS coefficients, consider `drop="first"` in the encoder; predictions can still be fitted without dropping categories. Penalty sizes depend on the target's units, especially for Elastic net.

### Fit and compare on validation

```python
fitted_models = {}
score_rows = []

for name, estimator in model_choices.items():
    candidate = clone(pipeline)
    candidate.set_params(model=clone(estimator))

    # Trees do not require numerical scaling.
    if name in ["Random forest", "XGBoost"]:
        candidate.set_params(preprocess__numeric__scale="passthrough")

    candidate.fit(X_train, y_train)
    train_pred = candidate.predict(X_train)
    valid_pred = candidate.predict(X_valid)

    fitted_models[name] = candidate
    score_rows.append({
        "model": name,
        "train_rmse": np.sqrt(mean_squared_error(y_train, train_pred)),
        "valid_rmse": np.sqrt(mean_squared_error(y_valid, valid_pred)),
        "valid_r2": r2_score(y_valid, valid_pred),
    })

comparison = pd.DataFrame(score_rows).sort_values("valid_rmse")
display(comparison)
```

`clone(pipeline)` creates an unfitted copy with the same settings, even if the original was fitted. Each candidate therefore learns its own preprocessing. `set_params(...)` updates settings and returns the estimator; changing a setting does not fit it. `model=...` replaces the model step, while `preprocess__numeric__scale` addresses a nested preprocessing step.

Select by validation RMSE; the test set has not been used. For returns, also compare with a zero-return forecast. RMSE is in the target's units; multiply by 10,000 to express log-return RMSE in basis points. A negative R² is possible.

### Final refit when using fixed candidate settings

Run this only after choosing the fixed-settings route. If you intend to tune, skip this block and use the search below before evaluating test.

```python
best_name = comparison.iloc[0]["model"]
final_model = clone(fitted_models[best_name])

X_development = pd.concat([X_train, X_valid], ignore_index=True)
y_development = pd.concat([y_train, y_valid], ignore_index=True)

final_model.fit(X_development, y_development)
test_predictions = final_model.predict(X_test)

print("Selected model:", best_name)
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, test_predictions)))
print("Test R2:", r2_score(y_test, test_predictions))
```

Do not inspect test results and then change the candidate settings.

Docs: [clone](https://scikit-learn.org/stable/modules/generated/sklearn.base.clone.html) · [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) · [Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html) · [RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html) · [XGBRegressor](https://xgboost.readthedocs.io/en/stable/python/python_api.html#xgboost.XGBRegressor).


<a id="pipeline-model-search"></a>

## Tune preprocessing and models with GridSearchCV

This is an alternative to the fixed-settings comparison. Start from the mixed-feature setup. The single chronological validation split below reuses already separated and purged `train` and `valid`; the test set stays outside the search. Each candidate fits its imputer, scaler, and encoder using only the fitting rows.

```python
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

X_development = pd.concat([X_train, X_valid], ignore_index=True)
y_development = pd.concat([y_train, y_valid], ignore_index=True)

fit_idx = np.arange(len(X_train))
valid_idx = np.arange(len(X_train), len(X_development))
validation_split = [(fit_idx, valid_idx)]

param_grid = [
    {"model": [DummyRegressor(strategy="mean")]},
    {"model": [LinearRegression()]},
    {
        "model": [Ridge()],
        "model__alpha": [0.1, 1.0, 10.0, 100.0],
    },
    {
        "model": [ElasticNet(max_iter=10000)],
        "model__alpha": [0.0001, 0.001],
        "model__l1_ratio": [0.2, 0.8],
    },
    {
        "model": [RandomForestRegressor(
            n_estimators=100, random_state=42, n_jobs=1
        )],
        "preprocess__numeric__scale": ["passthrough"],
        "model__max_depth": [4, 8],
        "model__min_samples_leaf": [5, 20],
    },
]
```

**Optional XGBoost grid:** append before constructing `search` if XGBoost is available.

```python
from xgboost import XGBRegressor

param_grid.append({
    "model": [XGBRegressor(
        objective="reg:squarederror", tree_method="hist",
        n_estimators=150, subsample=0.8, colsample_bytree=0.8,
        reg_lambda=1.0, random_state=42, n_jobs=1,
    )],
    "preprocess__numeric__scale": ["passthrough"],
    "model__max_depth": [2, 4],
    "model__learning_rate": [0.03, 0.1],
})
```

### Search, refit, and evaluate

```python
search = GridSearchCV(
    estimator=clone(pipeline),
    param_grid=param_grid,
    cv=validation_split,
    scoring="neg_root_mean_squared_error",
    refit=True,
    n_jobs=1,
    error_score="raise",
)
search.fit(X_development, y_development)

print("Best parameters:", search.best_params_)
print("Best validation RMSE:", -search.best_score_)

cv_results = pd.DataFrame(search.cv_results_)
cv_results["validation_rmse"] = -cv_results["mean_test_score"]
display(
    cv_results[["params", "validation_rmse", "rank_test_score"]]
    .sort_values("validation_rmse").head(8)
)

# This is the fitted winner, already refitted on train + validation.
best_pipeline = search.best_estimator_
development_predictions = best_pipeline.predict(X_development)
test_predictions = best_pipeline.predict(X_test)

print("Refitted development RMSE:", np.sqrt(
    mean_squared_error(y_development, development_predictions)
))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, test_predictions)))
print("Test R2:", r2_score(y_test, test_predictions))

encoded_names = best_pipeline.named_steps["preprocess"].get_feature_names_out()
print("Encoded feature count:", len(encoded_names))
```

| Name / option | Meaning |
|---|---|
| `model` | Replace the entire model step |
| `model__alpha` | Tune `alpha` on the model in that parameter dictionary |
| `preprocess__numeric__scale` | Replace the scale step inside the numerical branch |
| List of parameter dictionaries | Keep each model family's settings separate; avoid passing `alpha` to a forest |
| `cv=validation_split` | One fixed validation split using positional row indexes; not five-fold cross-validation |
| `neg_root_mean_squared_error` | Search maximizes the negative RMSE; negate scores for readable RMSE |
| `mean_test_score` | Historical API name for validation-fold scores; this is not the held-out test set |
| `refit=True` | Refit the selected settings on all supplied development rows, including validation |
| `best_estimator_` | The fitted winning pipeline; the template passed to the search is not fitted by the search |
| `n_jobs=1` | Run sequentially; increase search workers if useful while keeping tree-model workers at 1 |

After refitting, predictions on the former validation rows are in-sample. Use the stored search score for validation performance and the untouched test set for the final estimate. The small grids are starting points; choose ranges using training/validation evidence. If `numeric_columns` is empty, remove the numerical-scale parameter from the tree grids.

For independent observations, a shuffled `KFold` can replace this split. For time series, use chronological folds with label-horizon purging; an integer `cv=5` does not supply those safeguards. XGBoost early stopping is intentionally omitted: a raw `eval_set` does not automatically receive the pipeline's preprocessing.

Docs: [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) · [Pipeline parameters](https://scikit-learn.org/stable/modules/compose.html#access-to-nested-parameters) · [XGBoost sklearn interface](https://xgboost.readthedocs.io/en/stable/python/sklearn_estimator.html).


<a id="trade-outliers"></a>

## Find unusually large trade prices and quantities

Start by inspecting the largest values, then compare each trade with typical values for the same stock. These examples use the original `trade` DataFrame and leave it unchanged.

**1. Inspect the largest values.** `nlargest(10, column)` returns the ten rows with the largest values in that column, in descending order. It retains the other columns so you can identify the stock and timestamp.

```python
print("Largest trade prices")
display(trade.nlargest(10, "trade_price"))

print("Largest trade quantities")
display(trade.nlargest(10, "trade_qty"))
```

Large values are candidates for investigation, not necessarily errors. A normally expensive stock can appear at the top of the price table.

**2. Flag extremely large prices within each stock.** Use a median because one extreme value can greatly inflate the mean. Exclude nonpositive prices from the reference median.

```python
positive_prices = trade["trade_price"].where(trade["trade_price"] > 0)
median_prices = positive_prices.groupby(trade["symbol"]).transform("median")

large_price = trade["trade_price"] > 5 * median_prices

print("Flagged price rows:", large_price.sum())
display(trade.loc[large_price].nlargest(10, "trade_price"))
```

`groupby(trade["symbol"])` computes a separate reference for each stock. `transform("median")` puts that stock's median alongside every original row, so the comparison has the same length and index as `trade`. `.where(... > 0)` replaces nonpositive values with NaN; the median skips them.

Here, `5` is an illustrative screening threshold: flag prices greater than five times the stock's median. If the median is 20, the cutoff is 100. A price of 99,999,999 is flagged. Avoid using five times the median across all stocks: a stock normally trading around 225 should be compared with its own usual price.

**3. Flag extremely large quantities separately.** Trade sizes can vary much more than prices, so use a separate threshold.

```python
positive_quantities = trade["trade_qty"].where(trade["trade_qty"] > 0)
median_quantities = positive_quantities.groupby(trade["symbol"]).transform("median")

large_quantity = trade["trade_qty"] > 100 * median_quantities

print("Flagged quantity rows:", large_quantity.sum())
display(trade.loc[large_quantity].nlargest(10, "trade_qty"))
```

The `100` multiplier is also illustrative. If a stock's median quantity is 300, this flags quantities greater than 30,000. A large trade can be legitimate, so a quantity flag alone is not a reason to delete it.

**4. Keep the flags for review.** `|` means either condition; `&` would require both.

```python
flagged_trades = trade.loc[large_price | large_quantity].copy()
flagged_trades["large_price"] = large_price.loc[flagged_trades.index]
flagged_trades["large_quantity"] = large_quantity.loc[flagged_trades.index]

print("Rows flagged for review:", len(flagged_trades))
display(flagged_trades.head(10))
```

Compare flagged prices with nearby trades or bid and ask quotes for the same stock. Review flagged quantities against that stock's size distribution and the data specification. The multipliers are screening choices, not universal rules: some flagged trades are valid, and smaller errors can go undetected. Check missing, non-finite, zero, and negative values separately.

These examples suit exploratory checks on one trading day. For multiple days, compare within each stock and day. If the flags feed a predictive model, estimate reference values from training or past data only.

**Behavior:**

- **Returns:** `.nlargest()`, `.where()` with its default options, `.transform('median')`, `.head()`, and `.copy()`. Arithmetic, comparisons, and Boolean operators produce calculated values or masks; they do not alter `trade`.
- **Creates / reads:** `.groupby()` creates a grouping object; `.loc[...]` selects data and `.index` reads labels. `.transform('median')` returns a Series aligned to the original rows. `.sum()` and `len()` return counts.
- **Updates:** `flagged_trades['large_price'] = ...` and `flagged_trades['large_quantity'] = ...` add columns to the copied `flagged_trades`, not to `trade`.
- **Displays:** `print(...)` and `display(...)` return `None`. A displayed `.nlargest()` or `.head()` result does not change or save the original table.


<a id="histograms"></a>

## Histograms

A histogram divides numeric values into ranges (**bins**). Each bar counts how many observations fall in that range.

**Example: one histogram.** Use this when you only want one plot. This example assumes `trade_clean` is already loaded and cleaned.

```python
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))
ax.hist(trade_clean["trade_price"], bins=50, edgecolor="white")
ax.set(xlabel="Trade price", ylabel="Number of trades")
plt.show()
```

- Use `plt.subplots()` **with an s**: it returns both `fig` and `ax` and accepts `figsize`.
- For one plot, `ax` is a single axes object. Use `ax.hist(...)`, not `ax[0].hist(...)`.
- `plt.subplot()` **without an s** returns only an axes object and does not accept `figsize`.

**Example: 48-hour Facebook shares.** Use `train` for target exploration; `.dropna()` excludes missing values. The two plots show the same articles on raw and log-transformed scales.

```python
import numpy as np
import matplotlib.pyplot as plt

shares = train["TS144"].dropna()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(shares, bins=50, edgecolor="white")
axes[0].set_title("48-hour Facebook shares")
axes[0].set_xlabel("Shares")
axes[0].set_ylabel("Number of articles")

axes[1].hist(np.log1p(shares), bins=50, edgecolor="white")
axes[1].set_title("48-hour Facebook shares: log-transformed")
axes[1].set_xlabel("log(1 + shares)")
axes[1].set_ylabel("Number of articles")

plt.tight_layout()
plt.show()
```

| Code / option | Meaning | Behavior |
|---|---|---|
| `.hist(values, bins=50)` | Divide the value range into 50 equal-width bins and count observations | **Updates** axes; **returns** counts, bin edges, and artists |
| `edgecolor="white"` | Draw white borders between bars | Option controls the artists created by hist; input data unchanged |
| `plt.subplots(1, 2, figsize=(10, 4))` | One row, two plots; figure size in inches | **Creates and returns** figure and axes |
| `axes[0]` / `axes[1]` | Left / right plot | **Reads** an existing axes object |
| `.set_xlabel()` / `.set_ylabel()` | Label each axis | **Updates** axes text; **returns** a Text object |
| `plt.tight_layout()` | Adjust spacing around plots and labels | **Updates** figure layout; returns `None` |
| `plt.show()` | Display the figure | **Displays** figure; returns `None` |

**Figure size — `figsize=(width, height)`:** Sets the size of the entire figure in inches.

| Example | Figure size |
|---|---|
| `figsize=(6, 6)` | 6 inches wide, 6 inches tall; square |
| `figsize=(10, 4)` | 10 inches wide, 4 inches tall; wide and short |
| `figsize=(5, 8)` | 5 inches wide, 8 inches tall; narrow and tall |

With `plt.subplots(1, 2, figsize=(10, 4))`, the **two plots together** occupy one figure 10 inches wide and 4 inches tall, including labels and spacing.

Fewer bins give a broader summary; more bins show finer detail and can look noisy. Bins are equal-width on each plot's own scale, so bar heights are not directly comparable across the two plots.

**Supporting note — `log1p`:** `np.log1p(x)` means `ln(1 + x)`. For nonnegative share counts, it handles zero (`log1p(0) = 0`) and compresses large values, helping reveal the distribution among smaller counts. This transforms the x-values; the y-axis still counts articles.

**Interpretation:** Look for concentration near zero, a long right tail, or multiple peaks. After inspecting your plot, add a one- or two-sentence caption describing what it measures and the main observed pattern.

**Behavior:** Plotting can change a chart and return an object at the same time.

- `plt.subplots(...)` **creates and returns** `(fig, ax)` or `(fig, axes)`. `plt.subplot(...)` selects or creates an axes and returns it. `axes[0]` only retrieves an existing axes.
- `ax.hist(...)` **updates** the axes by adding histogram bars and **returns** counts, bin edges, and plot artists. No assignment is needed to draw the bars. Assign its return values only if you want to inspect or reuse them.
- `ax.set(...)` **updates** axes settings and **returns** setter results. `.set_title()`, `.set_xlabel()`, and `.set_ylabel()` update the displayed text and return Text objects.
- `plt.tight_layout()` **updates** the current figure's layout and returns `None`. `plt.show()` **displays** the figure and returns `None`. Neither changes the source DataFrame.
- `train['TS144'].dropna()` **returns** a filtered Series; `train` stays unchanged. `np.log1p(shares)` **returns** transformed values; `shares` stays unchanged. `figsize`, `bins`, and `edgecolor` are configuration arguments, not separate modifying operations.

The same distinction applies to the volume chart: `ax.bar(...)` **updates** the axes and **returns** a BarContainer; `ax.tick_params(...)` **updates** ticks/labels and returns `None`. These chart changes do not require assignment back to `ax`.


<a id="t-sf"></a>

## Student's t-distribution: `t.sf()` and p-values

`from scipy.stats import t` imports the Student's t-distribution object. Keep the name `t` for that object; use `t_stat` for a calculated statistic.

**`sf` means survival function:** `t.sf(x, df)` returns the probability to the right of `x` under a t-distribution with `df` degrees of freedom.

$$
\operatorname{sf}(x)=P(T>x), \qquad T\sim t_{\mathrm{df}}.
$$

| Code | Meaning |
|---|---|
| `t.sf(x, df)` | Right-tail area: probability that `T > x` |
| `t.cdf(x, df)` | Left-tail area: probability that `T <= x` |
| `2 * t.sf(abs(t_stat), df)` | Two-sided p-value: probability of a statistic at least as far from zero in either direction |

Mathematically, `sf(x) = 1 - cdf(x)`. Use `sf` directly because it is more numerically reliable for very small right-tail probabilities.

**Example: a two-sided one-sample t-test.** With 46 valid returns, the degrees of freedom are `46 - 1 = 45`. Suppose the observed t-statistic is 2:

```python
from scipy.stats import t

t_stat = 2.0
degrees_of_freedom = 46 - 1

right_tail = t.sf(abs(t_stat), df=degrees_of_freedom)
p_value = 2 * right_tail
print(f'Two-sided p-value: {p_value:.4f}')
```

The p-value is approximately **0.0516**. It combines `P(T < -2)` and `P(T > 2)`. A statistic of `-2` gives the same two-sided p-value.

- `abs(t_stat)` measures the distance from zero, regardless of sign.
- `t.sf(...)` gives the right-tail probability beyond that distance.
- Multiplying by 2 includes the equally large left tail because the t-distribution is symmetric.
- For a one-sample test of a mean, `df = n - 1`, where `n` counts valid observations. This degrees-of-freedom rule is specific to this test.

**Apply it to the stock-level results.** Assume `return_stats` already contains `count` and `t_stat` for each stock:

```python
from scipy.stats import t

degrees_of_freedom = return_stats['count'] - 1

return_stats['p_value'] = 2 * t.sf(
    return_stats['t_stat'].abs(),
    df=degrees_of_freedom
)

return_stats[['t_stat', 'p_value']].head()
```

SciPy calculates one probability per supplied statistic and matching degrees of freedom. Keep invalid tests as missing; fewer than two observations or zero sample standard deviation does not provide a usable ordinary t-statistic.

A p-value is the probability, under the null hypothesis and test assumptions, of a statistic at least as extreme as the observed one. It is **not** the probability that the null hypothesis is true. These are unadjusted p-values; correction across stocks, such as Bonferroni or Holm, is a separate step.

**Behavior:** `t.sf()`, `t.cdf()`, `.abs()`, and arithmetic **return** calculated values without changing their inputs. SciPy returns a scalar for scalar inputs and an array for array-like inputs. Assigning `return_stats['p_value'] = ...` **updates** that DataFrame by adding or replacing its p-value column.
